# SGC: propagasi fitur A²X + Logistic Regression
        **graph › node_classification** · notebook `sgc_logreg`

        **Kapan dipakai:** Titik awal GNN yang sangat kuat dan murah: tanpa jaringan saraf, hanya penghalusan fitur lewat graf + model linear.

        **Jebakan umum:** k terlalu besar menghaluskan berlebihan (semua simpul mirip). Pada fitur biner jarang, normalisasi baris + tanpa StandardScaler (dengan StandardScaler akurasi turun dari ~0.80 ke ~0.65 pada demo).

**Data demo:** Cora (2708 makalah, 5429 sitasi, 1433 fitur kata, 7 topik). Skenario semi-terawasi: hanya `PER_CLASS` simpul berlabel per kelas untuk latih, sisanya diuji; diulang pada beberapa seed.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Format data: `nodes.csv` (id, label, fitur...) dan `edges.csv` (source, target; arah diabaikan).

In [ ]:
GRAPH_DIR = "data_latihan/graph/cora"
MODEL_KEY = "sgc_logreg"
SEED = 42
OUT = out_dir("graph", "node_classification", MODEL_KEY); set_seed(SEED)
PER_CLASS, SEEDS = 20, [0, 1, 2]

## 2. Muat graf dan ringkasan

In [ ]:
from superdb.graph import load_graph
g = load_graph(GRAPH_DIR)
deg = np.asarray(g.A.sum(1)).ravel(); E = int(g.A.nnz // 2)
same = float((g.y[g.A.nonzero()[0]] == g.y[g.A.nonzero()[1]]).mean())
print(f"{g.n} simpul, {E} sisi, derajat rata-rata {deg.mean():.2f} (maks {int(deg.max())}), simpul terisolasi: {int((deg == 0).sum())}, fitur: {None if g.X is None else g.X.shape[1]}")
print(f"homofili sisi (porsi sisi yang menghubungkan simpul sekelas): {same:.2f}  -> {'graf homofili: tetangga informatif' if same > .6 else 'graf heterofili: tetangga kurang informatif'}")

## 3. Metode
Bentuk seragam: `fit_predict(g, tr, te)` mengembalikan probabilitas kelas simpul uji.

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from superdb.graph import structural_features, propagate, ppmi_embedding, label_propagation, train_gnn

def fit_predict(g, tr, te):
    X = g.X / np.maximum(g.X.sum(1, keepdims=True), 1)       # normalisasi baris
    Z = propagate(g.A_hat, X, k=2)            # fitur dirata-ratakan dengan tetangga 2 langkah
    m = LogisticRegression(C=100, max_iter=3000).fit(Z[tr], g.y[tr])
    return m.predict_proba(Z[te])

## 4. Evaluasi (rata-rata beberapa pembagian acak)

In [ ]:
from superdb.graph import split_per_class
from sklearn.metrics import accuracy_score, f1_score
import time; rows = []
for s in SEEDS:
    tr, te = split_per_class(g.y, PER_CLASS, seed=s); t0 = time.time(); P = fit_predict(g, tr, te); pred = P.argmax(1)
    rows.append({"seed": s, "accuracy": accuracy_score(g.y[te], pred), "f1_macro": f1_score(g.y[te], pred, average="macro"), "detik": time.time() - t0})
res = pd.DataFrame(rows); display(res.round(4)); metrics = {"accuracy_mean": float(res.accuracy.mean()), "accuracy_std": float(res.accuracy.std()), "f1_macro_mean": float(res.f1_macro.mean())}
print(f"akurasi = {res.accuracy.mean():.3f} ± {res.accuracy.std():.3f} (acak ≈ {1 / g.C:.2f}; mayoritas ≈ {np.bincount(g.y).max() / g.n:.2f})")

## 5. Confusion matrix (seed terakhir)

In [ ]:
plot_confusion(g.y[te], pred, labels=np.arange(g.C), path=OUT / "confusion.png"); plt.show()
save_json({"metrics": metrics, "per_seed": rows, "per_class": PER_CLASS}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan semua metode di folder ini; bandingkan juga dengan `features_logreg` untuk melihat nilai tambah graf.
- Label lebih banyak (`PER_CLASS` 50-100) mempersempit selisih antar metode.
- Gabungkan prediksi (blend) metode berbasis fitur dan berbasis struktur.